In [3]:
!pip uninstall -y bitsandbytes triton
!pip install -q \
  transformers==4.44.2 \
  accelerate==0.34.2 \
  peft==0.12.0 \
  datasets \
  sentence-transformers \
  rapidfuzz \
  rouge-score \
  scikit-learn

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 60.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.5 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip uninstall -y transformers tokenizers
!pip install transformers==4.51.3 tokenizers==0.21.1
!pip install -U accelerate peft bitsandbytes datasets huggingface_hub sentencepiece

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 81.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import transformers
print(transformers.__version__)

from transformers.cache_utils import SlidingWindowCache
print("SlidingWindowCache OK")

4.51.3
SlidingWindowCache OK


In [2]:
import os
os.environ["BITSANDBYTES_NOWELCOME"] = "1"

import json
import re
import time
import string
from dataclasses import dataclass
from typing import List, Dict, Optional
from collections import Counter

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training


class Args:
    original_data = "/kaggle/input/datasets/belalhesham/qa-data333333/qa_data.json"
    summarized_data = None

    model_name = "microsoft/Phi-4-mini-instruct"
    output_dir = "/kaggle/working/phi4_lora_output"

    seed = 42
    test_size = 0.15

    max_input_length = 768
    max_target_length = 48

    train_batch_size = 1
    eval_batch_size = 1
    grad_accum_steps = 8

    learning_rate = 2e-4
    weight_decay = 0.01
    warmup_ratio = 0.03
    num_epochs = 2

    lora_r = 8
    lora_alpha = 16
    lora_dropout = 0.05

    generation_max_new_tokens = 32
    threshold = 0.72

2026-05-10 21:41:35.290770: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778449295.514941     149 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778449295.583136     149 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778449296.125829     149 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778449296.125870     149 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778449296.125873     149 computation_placer.cc:177] computation placer alr

In [3]:
@dataclass
class Sample:
    knowledge: str
    question: str
    right_answer: str
    hallucinated_answer: str = ""
    source_type: str = "original"


def load_json_or_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
    except:
        pass

    records = []
    for line in text.splitlines():
        if line.strip():
            records.append(json.loads(line))
    return records


def build_samples(original_path, summarized_path=None):
    samples = []

    for r in load_json_or_jsonl(original_path):
        samples.append({
            "knowledge": str(r.get("knowledge", "")),
            "question": str(r.get("question", "")),
            "right_answer": str(r.get("right_answer", "")),
            "hallucinated_answer": str(r.get("hallucinated_answer", "")),
            "source_type": "original",
        })

    if summarized_path:
        for r in load_json_or_jsonl(summarized_path):
            samples.append({
                "knowledge": str(r.get("knowledge", "")),
                "question": str(r.get("question", "")),
                "right_answer": str(r.get("right_answer", "")),
                "hallucinated_answer": str(r.get("hallucinated_answer", "")),
                "source_type": "summarized",
            })

    return samples


def build_prompt(knowledge, question):
    return (
        "Answer using ONLY the given knowledge.\n\n"
        f"Knowledge:\n{knowledge}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )


def preprocess_dataset(dataset, tokenizer, max_input_length, max_target_length):
    def tokenize(batch):
        inputs = []
        masks = []
        labels = []

        for k, q, a in zip(batch["knowledge"], batch["question"], batch["right_answer"]):
            prompt = build_prompt(k, q)
            target = " " + a.strip() + tokenizer.eos_token

            prompt_ids = tokenizer(
                prompt,
                truncation=True,
                max_length=max_input_length
            )["input_ids"]

            target_ids = tokenizer(
                target,
                truncation=True,
                max_length=max_target_length,
                add_special_tokens=False
            )["input_ids"]

            input_ids = prompt_ids + target_ids
            label_ids = [-100]*len(prompt_ids) + target_ids

            inputs.append(input_ids)
            masks.append([1]*len(input_ids))
            labels.append(label_ids)

        return {
            "input_ids": inputs,
            "attention_mask": masks,
            "labels": labels
        }

    return dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)


class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids = []
        attention = []
        labels = []

        for f in features:
            pad_len = max_len - len(f["input_ids"])

            input_ids.append(f["input_ids"] + [self.tokenizer.pad_token_id]*pad_len)
            attention.append(f["attention_mask"] + [0]*pad_len)
            labels.append(f["labels"] + [-100]*pad_len)

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention),
            "labels": torch.tensor(labels)
        }


def main_train(args):
    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    print("Loading data...")
    df = pd.DataFrame(build_samples(args.original_data, args.summarized_data))

    train_df, eval_df = train_test_split(df, test_size=args.test_size, random_state=args.seed)
    eval_split_path = os.path.join(args.output_dir, "eval_split.csv")
    eval_df.reset_index(drop=True).to_csv(eval_split_path, index=False, encoding="utf-8-sig")

    print(f"Train: {len(train_df)} | Eval: {len(eval_df)}")
    print("Saved eval split to:", eval_split_path)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Loading Phi-4...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
        quantization_config=bnb_config,
        attn_implementation="eager",
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    # 🔥 مهم: طباعة أسماء layers
    print("\nInspecting model layers...")
    linear_names = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            linear_names.append(name)

    print(linear_names[:30])

    # ✅ LoRA target modules: auto-detect to support Phi-4 / Phi-4-mini safely
    available_linear_names = set(name.split(".")[-1] for name in linear_names)

    if "qkv_proj" in available_linear_names:
        target_modules = ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"]
    else:
        target_modules = [
            m for m in ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
            if m in available_linear_names
        ]

    if len(target_modules) == 0:
        raise ValueError(f"No valid LoRA target modules found. First linear layers: {linear_names[:30]}")

    print("LoRA target modules:", target_modules)

    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        target_modules=target_modules,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    train_ds = Dataset.from_pandas(train_df)
    eval_ds = Dataset.from_pandas(eval_df)

    train_tok = preprocess_dataset(train_ds, tokenizer, args.max_input_length, args.max_target_length)
    eval_tok = preprocess_dataset(eval_ds, tokenizer, args.max_input_length, args.max_target_length)

    collator = CausalLMCollator(tokenizer)

    # Works with both older and newer Transformers versions
    import inspect
    ta_kwargs = dict(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.train_batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum_steps,
        num_train_epochs=args.num_epochs,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        warmup_ratio=args.warmup_ratio,
        fp16=True,
        logging_steps=20,
        save_strategy="epoch",
        report_to="none",
    )
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
        ta_kwargs["eval_strategy"] = "epoch"
    else:
        ta_kwargs["evaluation_strategy"] = "epoch"

    training_args = TrainingArguments(**ta_kwargs)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        tokenizer=tokenizer,
        data_collator=collator,
    )

    print("Training...")
    trainer.train()

    save_path = os.path.join(args.output_dir, "phi4_lora")
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    return save_path, None


ADAPTER_DIR, _ = main_train(Args)

Loading data...
Train: 8500 | Eval: 1500
Saved eval split to: /kaggle/working/phi4_lora_output/eval_split.csv


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

/tmp/ipykernel_149/3932234207.py:234: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Epoch,Training Loss,Validation Loss
1,0.093100,0.184515


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


In [4]:
import re
import string
import time
from collections import Counter

def normalize_text(text):
    text = "" if text is None else str(text)
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

def exact_match(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def token_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

try:
    from rapidfuzz import fuzz

    def fuzzy_score(pred, gold):
        return fuzz.ratio(normalize_text(pred), normalize_text(gold)) / 100
except:
    from difflib import SequenceMatcher

    def fuzzy_score(pred, gold):
        return SequenceMatcher(None, normalize_text(pred), normalize_text(gold)).ratio()

try:
    from rouge_score import rouge_scorer
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

    def rouge_l_score(pred, gold):
        return rouge.score(str(gold), str(pred))["rougeL"].fmeasure
except:
    def rouge_l_score(pred, gold):
        return token_f1(pred, gold)

try:
    from sentence_transformers import SentenceTransformer, util
    sem_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def semantic_similarity(pred, gold):
        emb = sem_model.encode([str(pred), str(gold)], convert_to_tensor=True)
        return float(util.cos_sim(emb[0], emb[1]).item())
except:
    def semantic_similarity(pred, gold):
        return fuzzy_score(pred, gold)

def aggregate_score(pred, gold):
    em = exact_match(pred, gold)
    f1 = token_f1(pred, gold)
    fuzzy = fuzzy_score(pred, gold)
    rouge_l = rouge_l_score(pred, gold)
    sem = semantic_similarity(pred, gold)

    agg = (
        0.20 * em +
        0.25 * f1 +
        0.15 * fuzzy +
        0.15 * rouge_l +
        0.25 * sem
    )

    return {
        "exact_match": em,
        "token_f1": f1,
        "fuzzy_score": fuzzy,
        "rouge_l_f1": rouge_l,
        "semantic_similarity": sem,
        "aggregate_score": agg,
    }

def load_model_for_eval(base_model_name, adapter_dir):
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
        quantization_config=bnb_config,
        attn_implementation="eager"
    )

    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()

    return model, tokenizer

def generate_answer(model, tokenizer, knowledge, question, max_new_tokens=32):
    prompt = build_prompt(knowledge, question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=Args.max_input_length,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=2,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

def evaluate_generation(adapter_dir):
    eval_path = os.path.join(Args.output_dir, "eval_split.csv")

    if os.path.exists(eval_path):
        df = pd.read_csv(eval_path)
        print("Loaded eval_split.csv:", len(df))
    else:
        print("eval_split.csv مش موجود، هنعمل sample يدوي 1500 من الداتا الأصلية")
        df = pd.DataFrame(build_samples(Args.original_data, None))
        df = df.sample(1500, random_state=Args.seed).reset_index(drop=True)

    print("Evaluation samples:", len(df))

    model, tokenizer = load_model_for_eval(Args.model_name, adapter_dir)

    results = []
    start = time.time()

    for i, row in df.iterrows():
        pred = generate_answer(
            model,
            tokenizer,
            row["knowledge"],
            row["question"],
            Args.generation_max_new_tokens
        )

        gold = row["right_answer"]
        scores = aggregate_score(pred, gold)

        result = row.to_dict()
        result["generated_answer"] = pred
        result.update(scores)

        results.append(result)

        if (i + 1) % 100 == 0:
            print(f"{i+1}/{len(df)} done")

    result_df = pd.DataFrame(results)

    print("Result columns:", result_df.columns.tolist())

    result_path = os.path.join(Args.output_dir, "phi4_eval_1500_results.csv")
    result_df.to_csv(result_path, index=False, encoding="utf-8-sig")

    summary = {
        "num_samples": int(len(result_df)),
        "accuracy_by_threshold": float((result_df["aggregate_score"] >= Args.threshold).mean()),
        "exact_match_accuracy": float(result_df["exact_match"].mean()),
        "mean_token_f1": float(result_df["token_f1"].mean()),
        "mean_fuzzy_score": float(result_df["fuzzy_score"].mean()),
        "mean_rouge_l_f1": float(result_df["rouge_l_f1"].mean()),
        "mean_semantic_similarity": float(result_df["semantic_similarity"].mean()),
        "mean_aggregate_score": float(result_df["aggregate_score"].mean()),
        "threshold_used": Args.threshold,
        "runtime_minutes": round((time.time() - start) / 60, 2),
        "results_csv": result_path,
    }

    print("\n===== FINAL RESULTS =====")
    for k, v in summary.items():
        print(f"{k}: {v}")

    print("\nSaved results to:", result_path)

    return result_df, summary

result_df, summary = evaluate_generation(ADAPTER_DIR)

print("\nSample:")
print("Q:", result_df.iloc[0]["question"])
print("GT:", result_df.iloc[0]["right_answer"])
print("Pred:", result_df.iloc[0]["generated_answer"])
print("Score:", result_df.iloc[0]["aggregate_score"])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded eval_split.csv: 1500
Evaluation samples: 1500


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

100/1500 done
200/1500 done
300/1500 done
400/1500 done
500/1500 done
600/1500 done
700/1500 done
800/1500 done
900/1500 done
1000/1500 done
1100/1500 done
1200/1500 done
1300/1500 done
1400/1500 done
1500/1500 done
Result columns: ['knowledge', 'question', 'right_answer', 'hallucinated_answer', 'source_type', 'generated_answer', 'exact_match', 'token_f1', 'fuzzy_score', 'rouge_l_f1', 'semantic_similarity', 'aggregate_score']

===== FINAL RESULTS =====
num_samples: 1500
accuracy_by_threshold: 0.8406666666666667
exact_match_accuracy: 0.8266666666666667
mean_token_f1: 0.9077966998756345
mean_fuzzy_score: 0.9270189482567995
mean_rouge_l_f1: 0.9118831043307167
mean_semantic_similarity: 0.9458342832090954
mean_aggregate_score: 0.9045763869926432
threshold_used: 0.72
runtime_minutes: 25.35
results_csv: /kaggle/working/phi4_lora_output/phi4_eval_1500_results.csv

Saved results to: /kaggle/working/phi4_lora_output/phi4_eval_1500_results.csv

Sample:
Q: The manager in which Mark Lazarus clashed